In [10]:
import pandas as pd


In [11]:
df = pd.read_csv('test.csv')

In [12]:
df

In [13]:
def sz50():
    # 获取上证 50 成分股
    rs = bs.query_sz50_stocks()
    print('query_sz50 error_code:' + rs.error_code)
    print('query_sz50  error_msg:' + rs.error_msg)

    # 打印结果集
    sz50_stocks = []
    while (rs.error_code == '0') & rs.next():
        # 获取一条记录，将记录合并在一起
        sz50_stocks.append(rs.get_row_data())
    result = pd.DataFrame(sz50_stocks, columns=rs.fields)

    return result

In [14]:
import baostock as bs

bs.login()

sz50_stocks = sz50()


In [15]:
sz50_stocks

In [31]:
code = sz50_stocks['code'][0]


def get_stock(code: str):
    bs.login()

    rs = bs.query_history_k_data_plus(
        code=code,
        fields="date, code, open, high, low, close",
        start_date="2024-09-19",
        frequency="d",
        adjustflag="3"
    )

    # 打印结果集
    data_list = []
    while (rs.error_code == '0') & rs.next():
        # 获取一条记录，将记录合并在一起
        data_list.append(rs.get_row_data())

    result = pd.DataFrame(data_list, columns=rs.fields)

    bs.logout()

    return result.tail(1)

In [32]:
data = get_stock(code)

In [33]:
data

In [93]:
import numpy as np
import pandas as pd
import baostock as bs

from read_all_stock import sz50

bs.login()

sz50_stocks = sz50()

real_price = []
last_price = []
for code in sz50_stocks['code']:
    rs = bs.query_history_k_data_plus(
        code=code,
        fields="date, code, open, high, low, close",
        start_date="2024-09-10",
        frequency="d",
        adjustflag="3"
    )

    # 打印结果集
    data_list = []
    while (rs.error_code == '0') & rs.next():
        # 获取一条记录，将记录合并在一起
        data_list.append(rs.get_row_data())

    result = pd.DataFrame(data_list, columns=rs.fields)

    latest = result.tail(1)
    last = result.tail(2)['close'].to_list()[0]
    real_price.append(latest['close'].to_list()[0])
    last_price.append(last)

bs.logout()


In [94]:
real_price = [float(p) for p in real_price]
last_price = [float(p) for p in last_price]

In [95]:
real_price, last_price

In [96]:
df = pd.read_csv('test.csv', index_col=0)
# df['real_price'] = real_price

In [97]:
df = df.transpose()

In [98]:
df['real_price'] = real_price
df['latest'] = last_price 

In [99]:
df['real_rate'] = df['real_price'] / df['latest'] - 1.0

In [100]:
df

In [101]:
def determine_sign(row):
    if row['rate'] > 0 and row['real_rate'] > 0:
        return 1
    elif row['rate'] < 0 and row['real_rate'] < 0:
        return 1
    elif row['rate'] > 0 and row['real_rate'] < 0:
        return -1
    elif row['rate'] < 0 and row['real_rate'] > 0:
        return -1
    else:
        return 0
    
df['accurate'] = df.apply(determine_sign, axis=1)

In [103]:
df

In [105]:
# 统计 df['c'] 列中 1 和 -1 的个数
count_1 = (df['accurate'] == 1).sum()
count_neg1 = (df['accurate'] == -1).sum()

# 打印结果
print(f"1的个数: {count_1}")
print(f"-1的个数: {count_neg1}")

In [106]:
12/(12+35)